[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/06_inference_serving/06_inference_serving.ipynb)

# 06 · 推理服务：KV Cache、Batching、Speculative Decoding

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实模型维度上算 KV cache 显存、最大并发 batch、continuous batching 吞吐增益、投机解码加速。

**你将完成：**
1. KV cache 显存公式（看它如何超过权重本身）
2. 给定显存预算的最大并发序列数
3. 静态 vs continuous batching 吞吐模拟
4. speculative decoding 期望加速

> 数据：真实 Pythia config（KV/权重维度）。

## 0 · config 管线

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS={"pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
        "pythia-6.9b":"https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
        "pythia-12b":"https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json"}
def load_config(m):
    p=os.path.join(CACHE,f"{m}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(MODELS[m],p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
def param_count(c):
    L,h,V,I=c["L"],c["h"],c["V"],c["I"]; return 2*V*h+L*(4*h*h+4*h+2*h*I+(I+h)+4*h)+h
GB=1024**3
print('ok')

## 1 · KV cache 显存：会超过权重本身

每序列 KV = 2·L·S·h·bytes。算 pythia-6.9b 在长上下文 + 大并发下的 KV。

In [ ]:
def kv_bytes(cfg, S, batch=1, dtype_bytes=2):
    return 2*cfg["L"]*S*cfg["h"]*dtype_bytes*batch
cfg=load_config("pythia-6.9b"); P=param_count(cfg)
print(f"pythia-6.9b 权重(fp16) = {P*2/GB:.1f} GB")
for S,B in [(2048,1),(2048,32),(8192,32),(32768,16)]:
    kv=kv_bytes(cfg,S,B)
    print(f"  S={S:5d} batch={B:2d}: KV cache = {kv/GB:6.1f} GB  ({'> 权重!' if kv>P*2 else '< 权重'})")
print("\n=> 长上下文+高并发时 KV cache 吃的显存超过模型本身，这是 serving 的核心约束")

## 2 · 给定显存预算的最大并发

80GB 卡装下权重后，剩余显存能同时服务多少条 S 长的序列。

In [ ]:
def max_concurrent(cfg, card_gb, S, dtype_bytes=2):
    P=param_count(cfg); free=card_gb*GB - P*2          # 减去 fp16 权重
    per_seq=kv_bytes(cfg, S, 1, dtype_bytes)
    return max(0, int(free//per_seq))
for S in [2048, 8192, 32768]:
    n=max_concurrent(load_config("pythia-6.9b"), 80, S)
    print(f"80GB A100 服务 pythia-6.9b @ S={S:5d}: 最多 {n} 条并发序列")
print("\n=> 上下文越长，能并发的请求越少 -> 吞吐越低。KV 量化/GQA 直接放大并发")

## 3 · 静态 vs continuous batching 吞吐

不同请求长度不同。静态等最长的；continuous 完成即替换。模拟吞吐差异。

In [ ]:
rng=np.random.default_rng(0)
gen_lens=rng.integers(20, 200, size=64)   # 64 个请求各自生成长度
batch=8
def static_steps(lens, batch):
    total=0
    for i in range(0,len(lens),batch):
        total += lens[i:i+batch].max()    # 等这批最慢的
    return total
def continuous_steps(lens, batch):
    # 每步完成的槽位立刻补新请求；总步数≈总token/并发(满载)
    import heapq
    work=list(lens); active=[]; steps=0; i=0
    while i<batch and i<len(work): active.append(work[i]); i+=1
    while active:
        steps+=1; active=[a-1 for a in active]
        active=[a for a in active if a>0]
        while len(active)<batch and i<len(work): active.append(work[i]); i+=1
    return steps
s=static_steps(gen_lens,batch); c=continuous_steps(gen_lens,batch)
print(f"静态 batching 总步数 = {s}")
print(f"continuous batching 总步数 = {c}")
print(f"=> continuous 吞吐提升 {s/c:.2f}x（不等整批，槽位不空闲）")

## 4 · Speculative decoding 期望加速

draft 猜 k 个，target 一次验证。接受率 α 下期望接受长度与加速比。

In [ ]:
def expected_accept(alpha, k):
    # 接受到第一个失败为止，期望接受数 = sum_{i=1}^{k} alpha^i + ...（几何）
    # E[接受token数] = (1-alpha^(k+1))/(1-alpha) - 1 ，再 +1 个(失败处重采样也产1个)
    e = sum(alpha**i for i in range(1,k+1))   # 期望接受的 draft token 数
    return e + 1                              # +1: 失败处 target 仍产出 1 个
for alpha in [0.6, 0.8, 0.9]:
    for k in [4, 8]:
        tok=expected_accept(alpha,k)
        print(f"接受率 α={alpha} k={k}: 每次 target 前向期望产 {tok:.2f} tokens -> 约 {tok:.1f}x 加速")
print("\n=> draft 猜得越准(α高)、加速越大；且输出分布与直接解码完全一致(无损)")

## 5 · GQA / MQA 如何缩小 KV cache

KV cache 正比于 KV 头数。MHA 每个注意力头都有独立的 K/V；GQA 让若干 query 头共享一组 KV 头；MQA 是极端版（所有头共享 1 组）。下面在真实 pythia-6.9b 维度上对比三者的 KV 显存，看 GQA/MQA 把长上下文显存压几倍——这正是它直接放大并发请求数的原因。

In [ ]:
# GQA/MQA 如何缩小 KV cache：KV 头数从 heads 降到 kv_heads，cache 正比缩小。
# 在 pythia-6.9b 维度上对比 MHA / GQA / MQA。
cfg=load_config("pythia-6.9b"); n_heads=cfg["heads"]; head_dim=cfg["h"]//n_heads
def kv_bytes_grouped(cfg, S, batch, kv_heads, dtype_bytes=2):
    # KV 维度只随 kv_heads 走： 2 · L · S · (kv_heads·head_dim) · bytes · batch
    hd=cfg["h"]//cfg["heads"]
    return 2*cfg["L"]*S*(kv_heads*hd)*dtype_bytes*batch
S, B = 8192, 32
variants={"MHA (每头独立 KV)":n_heads, "GQA (8 组 KV 头)":8, "MQA (1 组 KV 头)":1}
print(f"pythia-6.9b @ S={S} batch={B}: KV cache 随 KV 头数缩小")
mha=kv_bytes_grouped(cfg,S,B,n_heads)
for name,kvh in variants.items():
    kv=kv_bytes_grouped(cfg,S,B,kvh)
    print(f"  {name:22s} kv_heads={kvh:2d}: {kv/GB:6.1f} GB  (MHA 的 {kv/mha:.3f}x)")
# 自检：GQA 比 MHA 小、MQA 最小，且比例 = kv_heads 之比
assert kv_bytes_grouped(cfg,S,B,8) < mha and kv_bytes_grouped(cfg,S,B,1) < kv_bytes_grouped(cfg,S,B,8)
assert abs(kv_bytes_grouped(cfg,S,B,8)/mha - 8/n_heads) < 1e-9
print("\n=> KV cache 正比于 KV 头数；GQA/MQA 通过共享 KV 头把长上下文显存压几倍，")
print("   直接放大可并发的请求数(上一节的 max_concurrent)")

---
## ✏️ 练习区

### ✏️ 练习 1：KV cache 显存

实现 `kv_cache_gb(cfg, S, batch, dtype_bytes)`：返回 KV cache 显存(GB)。
公式 `2·L·S·h·bytes·batch`。

In [ ]:
def kv_cache_gb(cfg, S, batch=1, dtype_bytes=2):
    # TODO: 2*L*S*h*dtype_bytes*batch / GB
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
cfg=load_config("pythia-6.9b")
kv = kv_cache_gb(cfg, 2048, 32)
assert kv > 0
# S 翻倍 -> KV 翻倍（线性）
assert abs(kv_cache_gb(cfg,4096,32) - 2*kv) < 1e-6
# int8(1字节) KV 是 fp16 的一半
assert abs(kv_cache_gb(cfg,2048,32,1) - kv/2) < 1e-6
print(f"练习 1 通过 ✓  6.9b @ S=2048 batch=32: KV={kv:.1f}GB")


### ✏️ 练习 2：最大并发序列

实现 `max_batch(cfg, card_gb, S)`：fp16 权重占用后，剩余显存能并发多少条 S 长序列。

In [ ]:
def max_batch(cfg, card_gb, S):
    # TODO: (card_gb*GB - 权重fp16) // 单序列KV
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
cfg=load_config("pythia-6.9b")
n_short=max_batch(cfg, 80, 2048); n_long=max_batch(cfg, 80, 32768)
assert n_short > n_long, "长上下文能并发的更少"
assert n_short > 0
print(f"练习 2 通过 ✓  80GB: S=2048 并发{n_short}条, S=32768 并发{n_long}条")


### ✏️ 练习 3：continuous batching 吞吐增益

实现 `throughput_gain(gen_lens, batch)`：返回 静态步数/continuous步数（>1 表示提升）。
可复用正文的 `static_steps`/`continuous_steps`。

In [ ]:
def throughput_gain(gen_lens, batch):
    # TODO: static_steps / continuous_steps
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
rng=np.random.default_rng(5)
lens=rng.integers(10, 300, size=100)
g=throughput_gain(lens, 8)
assert g >= 1.0, "continuous 不应比静态差"
# 长度方差越大，continuous 优势越明显
lens_var=np.array([10,500]*50)
assert throughput_gain(lens_var, 8) > 1.2
print(f"练习 3 通过 ✓  continuous 吞吐增益 {g:.2f}x")


### ✏️ 练习 4：投机解码期望加速

实现 `spec_speedup(alpha, k)`：返回每次 target 前向期望产出的 token 数
（接受的 draft token 数 + 1）。验证 α 越高加速越大。

In [ ]:
def spec_speedup(alpha, k):
    # TODO: sum(alpha^i for i in 1..k) + 1
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
assert spec_speedup(0.9, 8) > spec_speedup(0.6, 8), "接受率越高加速越大"
assert spec_speedup(0.8, 8) > spec_speedup(0.8, 2), "更长 draft 窗口(若接受率高)产出更多"
# α=0 时退化为普通解码(每次 1 token)
assert abs(spec_speedup(0.0, 4) - 1.0) < 1e-9
print(f"练习 4 通过 ✓  α=0.8 k=4 -> {spec_speedup(0.8,4):.2f} tokens/前向")


---
## 📖 参考答案

In [ ]:
# 练习 1
def kv_cache_gb(cfg, S, batch=1, dtype_bytes=2):
    return 2*cfg["L"]*S*cfg["h"]*dtype_bytes*batch/GB
print("练习 1 ✓")

In [ ]:
# 练习 2
def max_batch(cfg, card_gb, S):
    free=card_gb*GB - param_count(cfg)*2
    per=2*cfg["L"]*S*cfg["h"]*2
    return max(0, int(free//per))
print("练习 2 ✓")

In [ ]:
# 练习 3
def throughput_gain(gen_lens, batch):
    return static_steps(gen_lens,batch)/continuous_steps(gen_lens,batch)
print("练习 3 ✓")

In [ ]:
# 练习 4
def spec_speedup(alpha, k):
    return sum(alpha**i for i in range(1,k+1)) + 1
print("练习 4 ✓ —— 这些是估算 eval 推理成本的核心工具")